# Parse Log Files

In [1]:
import re
import pandas as pd 
import matplotlib as pyplot
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

In [2]:
# Compile regex pattern for training lines
_TRAINING_PATTERN = re.compile(
    r'Epoch: \[(\d+)\]\[(\d+)/(\d+)\]\s+'
    r'Time ([\d.]+) \(([\d.]+)\)\s+'
    r'Data ([\d.]+) \(([\d.]+)\)\s+'
    r'Xent ([\d.]+) \(([\d.]+)\)\s+'
    r'Htri ([\d.]+) \(([\d.]+)\)\s+'
    r'Acc ([\d.]+) \(([\d.]+)\)'
)


def _parse_training_line(line: str) -> Optional[Dict[str, float]]:
    """
    Parse a single training line into a dictionary.
    
    Args:
        line: Log line to parse
        
    Returns:
        Dict with all training metrics, or None if line doesn't match
    """
    match = _TRAINING_PATTERN.search(line)
    if not match:
        return None
    
    groups = match.groups()
    
    return {
        'epoch': int(groups[0]),
        'batch': int(groups[1]),
        'total_batches': int(groups[2]),
        'time_instant': float(groups[3]),
        'time_avg': float(groups[4]),
        'data_instant': float(groups[5]),
        'data_avg': float(groups[6]),
        'xent_instant': float(groups[7]),
        'xent_avg': float(groups[8]),
        'htri_instant': float(groups[9]),
        'htri_avg': float(groups[10]),
        'acc_instant': float(groups[11]),
        'acc_avg': float(groups[12]),
    }

In [3]:
def _parse_test_results(log_text: str) -> Dict[str, float]:
    """
    Parse test results section from log text.
    
    Args:
        log_text: Full log text to search
        
    Returns:
        Dict mapping metric names to percentage values
    """
    results = {}
    
    # Parse mAP
    map_pattern = r'mAP:\s+([\d.]+)%'
    map_match = re.search(map_pattern, log_text)
    if map_match:
        results['mAP'] = float(map_match.group(1))
    
    # Parse CMC ranks
    rank_pattern = r'Rank-(\d+)\s+:\s+([\d.]+)%'
    rank_matches = re.finditer(rank_pattern, log_text)
    for match in rank_matches:
        rank_name = f'Rank-{match.group(1)}'
        rank_value = float(match.group(2))
        results[rank_name] = rank_value
    
    return results

In [4]:
def _create_dataframes(
    training_data: List[Dict[str, float]], 
    test_data: Dict[str, float]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Convert parsed data to Pandas DataFrames.
    
    Args:
        training_data: List of training batch dicts
        test_data: Dict of test metric names to values
        
    Returns:
        Tuple of (training_df, test_results_df)
    """
    # Create training DataFrame
    if training_data:
        training_df = pd.DataFrame(training_data)
        # Ensure correct dtypes
        training_df['epoch'] = training_df['epoch'].astype('int64')
        training_df['batch'] = training_df['batch'].astype('int64')
        training_df['total_batches'] = training_df['total_batches'].astype('int64')
    else:
        # Empty DataFrame with correct schema
        training_df = pd.DataFrame(columns=[
            'epoch', 'batch', 'total_batches',
            'time_instant', 'time_avg',
            'data_instant', 'data_avg',
            'xent_instant', 'xent_avg',
            'htri_instant', 'htri_avg',
            'acc_instant', 'acc_avg'
        ])
    
    # Create test results DataFrame
    if test_data:
        test_df = pd.DataFrame([
            {'metric': metric, 'value': value}
            for metric, value in test_data.items()
        ])
    else:
        # Empty DataFrame with correct schema
        test_df = pd.DataFrame(columns=['metric', 'value'])
    
    return training_df, test_df

In [5]:
def parse_training_log(log_input: Union[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse vehicle re-identification training log.
    
    Extracts training batch metrics and test results into Pandas DataFrames.
    
    Args:
        log_input: A file path (str/Path)
        
    Returns:
        Tuple of (training_df, test_results_df)
        - training_df: DataFrame with columns [epoch, batch, total_batches, 
          time_instant, time_avg, data_instant, data_avg, xent_instant, 
          xent_avg, htri_instant, htri_avg, acc_instant, acc_avg]
        - test_results_df: DataFrame with columns [metric, value]
    """
    # Assume log_input is a file path
    with open(log_input, 'r') as f:
        log_text = f.read()

    training_data = []
    for line in log_text.splitlines():
        parsed = _parse_training_line(line)
        if parsed is not None:
            training_data.append(parsed)
    
    # Parse test results
    test_data = _parse_test_results(log_text)
    
    # Create DataFrames
    training_df, test_df = _create_dataframes(training_data, test_data)
    
    return training_df, test_df

In [13]:
log_input = "logs/mobilenet_v3_small-veri/log_train.txt"

training_df, test_df = parse_training_log(log_input)

In [14]:
training_df.head()

,epoch,batch,total_batches,time_instant,time_avg,data_instant,data_avg,xent_instant,xent_avg,htri_instant,htri_avg,acc_instant,acc_avg
0,1,10,464,0.035,0.638,0.0001,0.0533,6.3351,6.3678,0.1633,0.2415,1.56,0.78
1,1,20,464,0.034,0.338,0.0001,0.0267,6.3280,6.3442,0.0165,0.2192,1.56,0.70
2,1,30,464,0.088,0.242,0.0530,0.0227,6.3761,6.3159,0.1471,0.2198,0.00,0.68
3,1,40,464,0.035,0.193,0.0002,0.0200,6.1680,6.2918,0.0491,0.1989,3.12,0.98
4,1,50,464,0.095,0.165,0.0605,0.0192,6.2123,6.2639,0.1712,0.1955,1.56,1.25


In [15]:
test_df.head()

,metric,value
0,mAP,44.1
1,Rank-1,80.2
2,Rank-5,90.0
3,Rank-10,93.7
4,Rank-20,95.9


In [ ]:
# Create output directory if it doesn't exist
import os

output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

# Build filenames with optional prefix
experiment_name = "mobilenet_v3_small-veri"
training_filename = f"{experiment_name}_training.csv"
test_filename = f"{experiment_name}_test_results.csv"

# Save to CSV
training_path = os.path.join(output_dir, training_filename)
test_path = os.path.join(output_dir, test_filename)

training_df.to_csv(training_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"✓ Saved training data to: {training_path}")
print(f"✓ Saved test results to: {test_path}")

✓ Saved training data to: /results\mobilenet_v3_small-veri_training.csv
✓ Saved test results to: /results\mobilenet_v3_small-veri_test_results.csv
